# Group 38 adaptive query selection

In Colab, select **Runtime → Change runtime type → T4 GPU**. Push the implementation to the team repository before running. The checkout cell uses the current remote `main` commit by default. Paste a full commit SHA there to reproduce a specific run.

Run the cells in order. COCO and the checkpoints are staged under `/content`. Each run has its own output folder and Drive archive. Enable the last cell for the live upload demo.

In [ ]:
import importlib
import importlib.util
import platform
import subprocess
import sys

import torch
import torchvision

subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab and reconnect.")

gpu_name = torch.cuda.get_device_name(0)
print(f"Python {platform.python_version()}, torch {torch.__version__}, "
      f"torchvision {torchvision.__version__}, CUDA {torch.version.cuda}, GPU {gpu_name}")
if "T4" not in gpu_name:
    print(f"This run will measure {gpu_name}. Select a T4 runtime for the planned T4 comparison.")

def install_missing(packages):
    missing = [requirement for module, requirement in packages.items()
               if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
        importlib.invalidate_caches()

install_missing({"faster_coco_eval": "faster-coco-eval>=1.6.6", "pycocotools": "pycocotools"})

from faster_coco_eval.utils.pytorch import FasterCocoDetection
from pycocotools.cocoeval import COCOeval
import matplotlib
import numpy
from PIL import Image
import scipy
import tensorboard
import yaml

In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path

from google.colab import drive

REPO_URL = "https://github.com/Eswar2025/Object_detection_CV.git"
BRANCH = "main"
REVISION = ""  # Optional full Git commit SHA.
REPO = Path("/content/Object_detection_CV")

def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()

if REPO.exists():
    if git("remote", "get-url", "origin") != REPO_URL:
        raise RuntimeError(f"Unexpected Git remote in {REPO}")
    if git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError(f"Commit or stash tracked changes in {REPO} before updating.")
    refspec = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", refspec], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)

target = REVISION.strip().lower() or git("rev-parse", f"refs/remotes/origin/{BRANCH}")
if len(target) != 40 or any(char not in "0123456789abcdef" for char in target):
    raise ValueError("Use a full 40-character Git commit SHA in REVISION.")
if git("rev-parse", "--verify", f"{target}^{{commit}}") != target:
    raise RuntimeError(f"Commit {target} was not found in the team repo checkout.")
git("checkout", "--detach", target)
if git("rev-parse", "HEAD") != target:
    raise RuntimeError("The checked-out commit differs from the requested revision.")

PROJECT = REPO / "RT-DETR" / "rtdetrv2_pytorch"
for filename in ("query_budget_eval.py", "query_budget_report.py"):
    if not (PROJECT / "tools" / filename).is_file():
        raise FileNotFoundError(f"{filename} is missing at {target}. Push Group 38 and rerun this cell.")
git("cat-file", "-e", "df2a96c8:rtdetrv2_pytorch/src/zoo/rtdetr/rtdetrv2_decoder.py")
print(f"Verified Group 38 revision {target}")

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
from src.core import YAMLConfig

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/Group38")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = Path("/content/group38") / f"{stamp}-{target[:8]}"
RUN_DIR.mkdir(parents=True)
print(f"Run outputs will be written to {RUN_DIR}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import shutil
from zipfile import ZipFile

WEIGHTS = Path("/content/weights")
COCO = Path("/content/coco")
IMAGES = COCO / "val2017"
ANNOTATIONS = COCO / "annotations" / "instances_val2017.json"
COCO_BUCKET = "https://s3.amazonaws.com/images.cocodataset.org"

assets = [
    ("https://github.com/lyuwenyu/storage/releases/download/v0.1/rtdetrv2_r50vd_6x_coco_ema.pth",
     WEIGHTS / "rtdetrv2_r50vd_6x_coco_ema.pth"),
    ("https://github.com/lyuwenyu/storage/releases/download/v0.2/rtdetrv2_r18vd_120e_coco_rerun_48.1.pth",
     WEIGHTS / "rtdetrv2_r18vd_120e_coco_rerun_48.1.pth"),
]
images_ready = sum(1 for _ in IMAGES.glob("*.jpg")) == 5000
if not images_ready:
    assets.append((f"{COCO_BUCKET}/zips/val2017.zip", COCO / "val2017.zip"))
if not ANNOTATIONS.is_file():
    assets.append((f"{COCO_BUCKET}/annotations/annotations_trainval2017.zip",
                   COCO / "annotations_trainval2017.zip"))

def download(asset):
    url, path = asset
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file() and path.stat().st_size > 0:
        print(f"Using {path.name} from /content")
        return
    partial = path.with_suffix(path.suffix + ".part")
    subprocess.run(["curl", "--fail", "--location", "--silent", "--show-error", "--retry", "4",
                    "--retry-delay", "3", "--connect-timeout", "30", "--output", str(partial), url],
                   check=True)
    partial.replace(path)
    print(f"Downloaded {path.name} ({path.stat().st_size / 1e6:.1f} MB)")

with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(download, assets))

if not images_ready:
    with ZipFile(COCO / "val2017.zip") as archive:
        archive.extractall(COCO)
if not ANNOTATIONS.is_file():
    partial = ANNOTATIONS.with_suffix(".json.part")
    ANNOTATIONS.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(COCO / "annotations_trainval2017.zip") as archive:
        with archive.open("annotations/instances_val2017.json") as source, partial.open("wb") as dest:
            shutil.copyfileobj(source, dest)
    partial.replace(ANNOTATIONS)
assert sum(1 for _ in IMAGES.glob("*.jpg")) == 5000 and ANNOTATIONS.is_file()
print("Both checkpoints and all 5,000 COCO val2017 JPEGs and instance annotations are ready.")

In [ ]:
EVAL_COMMAND = [
    sys.executable, "-u", "tools/query_budget_eval.py",
    "--images", str(IMAGES), "--annotations", str(ANNOTATIONS), "--weights-dir", str(WEIGHTS),
    "--seed", "38", "--calibration", "64", "--test", "256", "--out", str(RUN_DIR),
]
subprocess.run([*EVAL_COMMAND, "--check-only"], cwd=PROJECT, check=True)

In [ ]:
# The pilot can freeze the 128-image prefix before any held-out inference.
subprocess.run(EVAL_COMMAND, cwd=PROJECT, check=True)

In [ ]:
subprocess.run([sys.executable, "-u", "tools/query_budget_report.py", "--run", str(RUN_DIR)],
               cwd=PROJECT, check=True)

In [ ]:
import hashlib
import shutil

from IPython.display import Image as DisplayImage, display

archive_path = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
drive_path = DRIVE_DIR / archive_path.name
partial = drive_path.with_suffix(".zip.part")
shutil.copyfile(archive_path, partial)

def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

if digest(archive_path) != digest(partial):
    raise IOError("Drive copy failed the SHA-256 verification.")
with ZipFile(partial) as archive:
    corrupt = archive.testzip()
    if corrupt is not None:
        raise IOError(f"Invalid archive entry {corrupt}")
partial.replace(drive_path)
print(f"Verified and saved {drive_path} ({drive_path.stat().st_size / 1e6:.1f} MB)")
display(DisplayImage(filename=str(RUN_DIR / "map_vs_latency.png")))
display(DisplayImage(filename=str(RUN_DIR / "adaptive_budget_counts.png")))

In [ ]:
RUN_DEMO = False  # Set to True after the report has been backed up.

if RUN_DEMO:
    install_missing({"gradio": "gradio"})
    import gradio as gr
    from tools.query_budget_report import make_upload_demo

    compare = make_upload_demo(RUN_DIR, WEIGHTS)
    gr.Interface(
        fn=compare,
        inputs=gr.Image(type="pil", label="Upload a photo"),
        outputs=[gr.Image(type="pil", label="Stock and adaptive"), gr.Textbox(label="Queries and time")],
        title="Group 38 · 300 queries vs. an adaptive budget",
    ).launch(inline=True)